# PP-OCRv6-derived price reader (custom)

This separate family is the measured price candidate. It uses a 128×320 BGR input, CTC decoding, reconstructed training-time BN, and a training-only PARSeq Tiny auxiliary reader. The auxiliary reader is discarded for inference. This is a local training recipe, not an official PP-OCRv6 model option.

Historical results:187/206 numeric agreement;131/144 on the AI-reviewed clear-price subset versus Florence133/144. The144 examples are not human-certified. Historical TensorRT1.319ms/670MiB was measured on RTX5090 with a specific environment; rebuilding requires accuracy and timing checks. Changing the dataset or recipe can change results.

Install `pip install -e ".[ppocr,onnx]"` for training/conversion and add `ppocr-tensorrt` for NVIDIA deployment. No pretrained/fine-tuned weights are bundled. Set dataset_root in configs/ppocr_price.yaml to Image/TXT train/val folders. Selection uses TRAIN only, then full-fit starts fresh from official initialization for the selected epoch count. External val never selects a checkpoint. The default exact-selection recipe preserves the historical policy; numeric_then_exact is an explicit alternative, not the same experiment.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from parseq_ocr.ppocr_config import load_config
cfg = load_config(ROOT / 'configs/ppocr_price.yaml')
{k: str(v) for k, v in cfg.items()}

Run these steps only after checking your own paths. Official pretrained hashes are verified; local conversion keeps the original math before the explicit height/vocabulary adaptation. Fit-only images initialize BN statistics; holdout images never calibrate selection BN. Labels retain punctuation, spaces and cents and are never truncated. The current auxiliary decoder supports at most12 characters.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from parseq_ocr.ppocr_training import train
    checkpoint = train(cfg)
    print(checkpoint)
else:
    print('Training disabled; edit config and opt in to run.')

In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    from parseq_ocr.ppocr_export import export_onnx, build_engine
    checkpoint = cfg['output_dir'] / 'full_fit/model.pt'
    deployment = export_onnx(checkpoint, cfg['output_dir'] / 'deployment')
    build_engine(deployment)

Equivalent CLI:

```bash
python -m parseq_ocr.ppocr_cli prepare --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli train --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli evaluate --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli export --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli build --config configs/ppocr_price.yaml
python -m parseq_ocr.ppocr_cli evaluate --config configs/ppocr_price.yaml --runtime tensorrt
```

Batch1 through32 is supported by the PP TensorRT profile. CTC collapses repeats before removing blank0; no country grammar or price substring extraction is applied. Decimal numeric scoring uses the entire OCR string. Report exact and numeric metrics together, and visually inspect cents placement separately. Benchmark image decoding consistently; predecoded RGB-to-text timing differs from file-to-text timing. WATERec orientation is a separate ALPR option and does not apply to this reader.


```bash
# File-to-text timing (batch1); batch32 reports amortized throughput.
python -m parseq_ocr.ppocr_cli benchmark --config configs/ppocr_price.yaml --runtime tensorrt --batch-size 1
python -m parseq_ocr.ppocr_cli benchmark --config configs/ppocr_price.yaml --runtime tensorrt --batch-size 32 --predecoded
```
